# 🐴 netkeiba スクレイピング → Supabase

**使い方：**
1. セル1でライブラリをインストール
2. セル2でSupabaseの接続情報を入力
3. セル3でスクレイピング関数を定義
4. セル4で取得期間を指定して実行

**注意：** 無料版Colabは最大12時間で切断されます。  
大量データは年ごとに分割して実行してください。

| 期間 | 目安時間 |
|------|----------|
| 1ヶ月 | 約40分 |
| 6ヶ月 | 約4時間 |
| 1年 | 約8時間 |

In [ ]:
# ① ライブラリインストール
!pip install requests beautifulsoup4 supabase -q

In [ ]:
# ② Supabase接続情報（ここを編集）
SUPABASE_URL = "https://infypumigexmpdmijhnx.supabase.co"
SUPABASE_KEY = "eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9..."  # service_role keyを貼り付け

from supabase import create_client
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Supabase接続OK")

In [ ]:
# ③ スクレイピング関数定義
import re
import time
import random
import logging
from datetime import datetime, timedelta
import requests
from bs4 import BeautifulSoup

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

BASE_DB   = "https://db.netkeiba.com"
BASE_RACE = "https://race.netkeiba.com"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "ja,en-US;q=0.7,en;q=0.3",
    "Referer": "https://db.netkeiba.com/",
}

session = requests.Session()
session.headers.update(HEADERS)


def fetch(url: str):
    try:
        time.sleep(random.uniform(2, 4))  # Colab個人IPなので少し速くできる
        res = session.get(url, timeout=20)
        res.encoding = "EUC-JP"
        if res.status_code == 403:
            logger.warning(f"403: {url} - skip")
            return None
        if res.status_code == 404:
            return None
        res.raise_for_status()
        return BeautifulSoup(res.text, "html.parser")
    except Exception as e:
        logger.error(f"fetch error: {url} -> {e}")
        return None


def upsert(table, data):
    try:
        supabase.table(table).upsert(data).execute()
    except Exception as e:
        logger.error(f"upsert {table}: {e}")


def get_race_id_list(date: str):
    """指定日のレースID一覧（db.netkeiba.comから取得）"""
    url = f"{BASE_RACE}/top/race_list_sub.html?kaisai_date={date}"
    soup = fetch(url)
    if not soup:
        return []
    race_ids = []
    for a in soup.find_all("a", href=True):
        m = re.search(r"race_id=(\d{12})", a["href"])
        if m:
            race_ids.append(m.group(1))
    return list(set(race_ids))


def scrape_horse(horse_id: str):
    url = f"{BASE_DB}/horse/{horse_id}/"
    soup = fetch(url)
    if not soup:
        return
    profile = {}
    tbl = soup.find("table", class_="db_prof_table")
    if tbl:
        for tr in tbl.find_all("tr"):
            th = tr.find("th")
            td = tr.find("td")
            if th and td:
                profile[th.text.strip()] = td.text.strip()
    title = soup.find("div", class_="horse_title")
    upsert("horses", {
        "horse_id":   horse_id,
        "horse_name": title.find("h1").text.strip() if title else "",
        "birth_date": profile.get("生年月日"),
        "sex":        profile.get("性別"),
        "coat_color": profile.get("毛色"),
    })
    ped = soup.find("table", class_="blood_table")
    if ped:
        cells = [td.text.strip() for td in ped.find_all("td")]
        if len(cells) >= 6:
            upsert("horse_pedigrees", {
                "horse_id":      horse_id,
                "father":        cells[0], "father_father": cells[1],
                "father_mother": cells[2], "mother":        cells[3],
                "mother_father": cells[4], "mother_mother": cells[5],
            })


def scrape_race(race_id: str):
    url = f"{BASE_DB}/race/{race_id}/"
    soup = fetch(url)
    if not soup:
        return

    # race_idから日付を確実に取得
    race_data = {
        "race_id":     race_id,
        "venue_code":  race_id[4:6],
        "race_date":   f"{race_id[0:4]}-{race_id[6:8]}-{race_id[8:10]}",
        "race_number": int(race_id[10:12]),
    }

    # レース名
    head = soup.find("div", class_="race_head_inner") or soup.find("div", class_="mainrace_data")
    if head:
        h1 = head.find("h1")
        if h1: race_data["race_name"] = h1.text.strip()

    # コース情報
    ct = soup.find("div", class_="race_data")
    if ct:
        t = ct.text
        if "芝" in t:   race_data["surface"] = "芝"
        elif "ダ" in t: race_data["surface"] = "ダート"
        elif "障" in t: race_data["surface"] = "障害"
        if "右" in t:    race_data["direction"] = "右"
        elif "左" in t:  race_data["direction"] = "左"
        m = re.search(r"(\d+)m", t)
        if m: race_data["distance"] = int(m.group(1))
        mw = re.search(r"天候:(\S+)", t)
        if mw: race_data["weather"] = mw.group(1)
        mt = re.search(r"馬場:(\S+)", t)
        if mt: race_data["track_condition"] = mt.group(1)

    upsert("races", race_data)

    # 出走・結果
    rt = soup.find("table", class_="race_table_01")
    if rt:
        for tr in rt.find_all("tr")[1:]:
            cols = tr.find_all("td")
            if len(cols) < 10: continue
            try:
                hh  = cols[3].find("a")["href"] if cols[3].find("a") else ""
                jh  = cols[6].find("a")["href"] if cols[6].find("a") else ""
                th2 = cols[7].find("a")["href"] if cols[7].find("a") else ""
                hid = re.search(r"/horse/(\w+)", hh)
                jid = re.search(r"/jockey/(\w+)", jh)
                tid = re.search(r"/trainer/(\w+)", th2)
                horse_id   = hid.group(1) if hid else None
                jockey_id  = jid.group(1) if jid else None
                trainer_id = tid.group(1) if tid else None
                if jockey_id:  upsert("jockeys",  {"jockey_id": jockey_id, "jockey_name": cols[6].text.strip()})
                if trainer_id: upsert("trainers", {"trainer_id": trainer_id, "trainer_name": cols[7].text.strip()})
                if horse_id:
                    ex = supabase.table("horses").select("horse_id").eq("horse_id", horse_id).execute()
                    if not ex.data:
                        scrape_horse(horse_id)
                fp = cols[0].text.strip()
                upsert("race_entries", {
                    "race_id": race_id, "horse_id": horse_id,
                    "jockey_id": jockey_id, "trainer_id": trainer_id,
                    "post_position":   int(cols[1].text.strip()) if cols[1].text.strip().isdigit() else None,
                    "horse_number":    int(cols[2].text.strip()) if cols[2].text.strip().isdigit() else None,
                    "finish_position": int(fp) if fp.isdigit() else None,
                    "finish_time":     cols[8].text.strip() if len(cols) > 8 else None,
                    "margin":          cols[9].text.strip() if len(cols) > 9 else None,
                    "odds":            float(cols[12].text.strip()) if len(cols) > 12 and cols[12].text.strip().replace(".","").isdigit() else None,
                    "popularity":      int(cols[13].text.strip()) if len(cols) > 13 and cols[13].text.strip().isdigit() else None,
                    "weight":          int(re.search(r"(\d+)", cols[14].text).group(1)) if len(cols) > 14 and re.search(r"(\d+)", cols[14].text) else None,
                    "weight_diff":     int(re.search(r"[+-]?\d+", cols[14].text.replace("(","").replace(")","")).group()) if len(cols) > 14 and re.search(r"[+-]?\d+", cols[14].text) else None,
                })
            except Exception as e:
                logger.warning(f"entry error: {race_id} -> {e}")

    # 払い戻し
    for pt in soup.find_all("table", class_="pay_table_01"):
        for tr in pt.find_all("tr"):
            cols = tr.find_all("td")
            if len(cols) < 3: continue
            try:
                bt      = cols[0].text.strip()
                combos  = [s.strip() for s in cols[1].text.strip().split("\n") if s.strip()]
                amounts = [s.strip() for s in cols[2].text.strip().split("\n") if s.strip()]
                pops    = [s.strip() for s in cols[3].text.strip().split("\n") if s.strip()] if len(cols) > 3 else []
                for i, combo in enumerate(combos):
                    upsert("payouts", {
                        "race_id": race_id, "bet_type": bt, "combination": combo,
                        "payout": int(amounts[i].replace(",","").replace("円","")) if i < len(amounts) else 0,
                        "popularity": int(pops[i].replace("番人気","")) if i < len(pops) else None,
                    })
            except Exception as e:
                logger.warning(f"payout error: {race_id} -> {e}")

    logger.info(f"done: {race_id}")


def scrape_date_range(start: str, end: str):
    """start, end: 'YYYYMMDD'"""
    # セッション初期化
    session.get(f"{BASE_DB}/", timeout=20)
    time.sleep(3)

    current = datetime.strptime(start, "%Y%m%d")
    end_dt  = datetime.strptime(end,   "%Y%m%d")
    total_days = (end_dt - current).days + 1
    done_days  = 0

    while current <= end_dt:
        ds = current.strftime("%Y%m%d")
        race_ids = get_race_id_list(ds)
        print(f"📅 {ds} | {len(race_ids)}レース | 進捗: {done_days}/{total_days}日")
        for race_id in race_ids:
            scrape_race(race_id)
        done_days += 1
        current += timedelta(days=1)

    print("✅ 完了！")

print("✅ 関数定義完了")

In [ ]:
# ④ 実行（期間を指定してください）
# 例: 直近1ヶ月
# scrape_date_range("20260401", "20260430")

# 例: 2025年前半
# scrape_date_range("20250101", "20250630")

# 例: 2025年全体
# scrape_date_range("20250101", "20251231")

# ▼ ここを書き換えて実行
scrape_date_range("20260401", "20260430")

In [ ]:
# ⑤ データ確認（取り込み後に実行）
for table in ["races", "race_entries", "horses", "payouts", "jockeys", "trainers"]:
    count = supabase.table(table).select("*", count="exact").execute()
    print(f"{table:15s}: {count.count:,} 件")